In [ ]:
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import time
import pickle
import pandas as pd
from IPython.display import clear_output
import mloop.interfaces as mli
import mloop.controllers as mlc
import mloop.visualizations as mlv
import gc
import os

In [ ]:
 # amplitude increment
class SIMCIM():
    #Initial parameters
    def __init__(self, J3, O,S,D, N,dt, tau, dim, sigma, zeta, alpha,  attempt_num,  device='cuda'):
        
        self.J3=torch.tensor(J3, device=device, dtype=torch.float)
        
        
        #Parameters of the algorithm (for adjusting)
        self.O=O
        self.S=S
        self.D=D
        self.N=N
        self.tau=tau
        self.zeta=zeta
        self.alpha=alpha
        self.zeta=zeta
        self.dt=dt
        self.sigma = sigma
        #Parameters of the algorithm (will not be changed)
        c_th=1
        self.attempt_num = attempt_num
        self.Jmax = torch.mean(torch.sum(torch.abs(self.J3),(1,2)))
        self.device = device
        self.dim=self.J3.size(0)
        self.tau=int(2*self.J3.size(0))
        #self.lastcuts=torch.zeros(batch)
    #amplitude increment
    
    def ampl_inc(self, c,p):
        return (p*c - self.zeta/self.Jmax*(3*torch.einsum('ijk,js,ks->is',(self.J3, c, c))))*self.dt + self.sigma*torch.randn_like(c)


    # pump ramp
    def pump(self):
        j = torch.arange(self.N, dtype = torch.float, device=self.device)
        arg = torch.tensor(self.S,dtype = torch.float, device=self.device)*(j/self.N-0.5)
        return self.Jmax/self.dim*self.O*(torch.tanh(arg) + self.D )
    

    
    
    # evolution of amplitudes
    # init_value -- init coupling
    # final_value -- final coupling
    # N -- number of time iterations
    # attempt_num -- number of runs 
    # J -- coupling matrix
    # b -- biases
    # O, S, D -- pump parameters
    # sigma -- sqrt(std) for random number
    # alpha -- momentum parameter
    # c_th -- restriction on amplitudes growth
    def evolve(self):
        
        #dimension of the arrays

        i=0
        #random
        random_attempt = np.random.randint(self.attempt_num)

        # initializing current amplitudes
        c = torch.zeros((self.dim, self.attempt_num),dtype=torch.float, device=self.device)
        dc_momentum = torch.zeros_like(c)
        spins_opt=torch.zeros_like(c)
        spins=torch.sign(c)
        dc=torch.zeros_like(c)

        # define pump array
        p = self.pump()
        H_opt=torch.einsum('ijk,is,js,ks->s',(self.J3, spins,spins, spins))
        data = []
        best_spins = None
#         t1=0
        #initial values of amplitude and dc
        
        
        # initializing moving average of amplitudes increment
        
        #c_all=torch.zeros((dim, attempt_num, N), dtype=torch.float)
        for i in range(1,self.N):
            

            dc = self.ampl_inc(c, p[i])
#             zeta=self.coupling(t1,rate, device=self.device)
            
            # calculating moving average of amplitudes increment
            dc_momentum = self.alpha*dc_momentum + (1-self.alpha)*dc
            c1=c+dc_momentum
            check1=(torch.abs(c1)<=1.0).float()

            c=c*(1-check1)+check1*c1
            # calculating possible values of amplitudes on the next 
            if i % 20 == 0:
                spins=torch.sign(c)
                data.append(spins.T.cpu())
                H = torch.einsum('ijk,is,js,ks->s',(self.J3, spins,spins, spins))
                check = (H<H_opt).type(torch.float)
                H_opt = check*H+(1-check)*H_opt
                spins_opt = check*spins + (1-check) * spins_opt
                #print(H_opt.min().item())
                best_idx = H_opt.argmin().item()
                best_spins = spins_opt.T[best_idx].cpu().numpy()
            
        spins = spins_opt
        self.lastcuts = H_opt 
       
        return best_spins, data

In [ ]:
def compute_energy_sparse(x, H1, H2, H3):

    energy = 0.0
    n = len(x)
    
    # Линейные
    energy += np.dot(x, H1)
    
    # Квадратичные (только верхний треугольник)
    for i in range(n):
        for j in range(i+1, n):
            if H2[i, j] != 0:
                energy += H2[i, j] * x[i] * x[j]
    
    # Кубические (только i < j < k)
    for i in range(n):
        for j in range(i+1, n):
            for k in range(j+1, n):
                if H3[i, j, k] != 0:
                    energy += H3[i, j, k] * x[i] * x[j] * x[k]
    
    return energy